In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Singularity Machine Learning - Classification: Uma Qiskit Function da Multiverse Computing
> **Note:** * As Qiskit Functions são um recurso experimental disponível apenas para usuários dos planos IBM Quantum&reg; Premium, Flex e On-Prem (via API da IBM Quantum Platform). Elas estão em status de pré-lançamento e sujeitas a alterações.

## Visão geral
Com a função "Singularity Machine Learning - Classification", você pode resolver problemas reais de aprendizado de máquina em hardware quântico sem precisar de expertise em computação quântica. Essa função de aplicação, baseada em métodos de ensemble, é um classificador híbrido. Ela utiliza métodos clássicos como boosting, bagging e stacking para o treinamento inicial do ensemble. Em seguida, algoritmos quânticos como o variational quantum eigensolver (VQE) e o quantum approximate optimization algorithm (QAOA) são empregados para aprimorar a diversidade, as capacidades de generalização e a complexidade geral do ensemble treinado.

Ao contrário de outras soluções de aprendizado de máquina quântico, essa função é capaz de lidar com conjuntos de dados em larga escala, com milhões de exemplos e features, sem ser limitada pelo número de qubits no QPU alvo. O número de qubits apenas determina o tamanho do ensemble que pode ser treinado. Ela também é altamente flexível e pode ser usada para resolver problemas de classificação em uma ampla gama de domínios, incluindo finanças, saúde e segurança cibernética.
Ela consegue consistentemente alta precisão em problemas classicamente desafiadores que envolvem conjuntos de dados de alta dimensionalidade, ruidosos e desbalanceados.
![Como funciona](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
Ela foi construída para:
1. Engenheiros e cientistas de dados em empresas que buscam aprimorar suas ofertas tecnológicas integrando aprendizado de máquina quântico em seus produtos e serviços,
2. Pesquisadores em laboratórios de pesquisa quântica que exploram aplicações de aprendizado de máquina quântico e querem aproveitar a computação quântica para tarefas de classificação, e
3. Estudantes e professores em instituições educacionais em cursos como aprendizado de máquina, que buscam demonstrar as vantagens da computação quântica.

O exemplo a seguir demonstra suas diversas funcionalidades, incluindo `create`, `list`, `fit` e `predict`, e demonstra seu uso em um problema sintético composto por dois semicírculos entrelaçados, um problema notoriamente desafiador devido à sua fronteira de decisão não linear.


## Descrição da função
Essa Qiskit Function permite que os usuários resolvam problemas de classificação binária usando o classificador de ensemble aprimorado quanticamente da Singularity. Por trás dos panos, ela usa uma abordagem híbrida para treinar classicamente um ensemble de classificadores no conjunto de dados rotulado e, em seguida, otimizá-lo para máxima diversidade e generalização usando o Quantum Approximate Optimization Algorithm (QAOA) nos QPUs da IBM&reg;. Por meio de uma interface amigável, os usuários podem configurar um classificador de acordo com seus requisitos, treiná-lo no conjunto de dados de sua escolha e usá-lo para fazer previsões em um conjunto de dados nunca visto anteriormente.

Para resolver um problema genérico de classificação:
1. Pré-processe o conjunto de dados e divida-o em conjuntos de treinamento e teste. Opcionalmente, você pode dividir ainda mais o conjunto de treinamento em conjuntos de treinamento e validação. Isso pode ser feito usando [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. Se o conjunto de treinamento estiver desbalanceado, você pode reamostrar para balancear as classes usando [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. Faça upload dos conjuntos de treinamento, validação e teste separadamente para o armazenamento da função usando o método `file_upload` do catálogo, passando o caminho relevante a cada vez.
4. Inicialize o classificador quântico usando a ação `create` da função, que aceita hiperparâmetros como o número e os tipos de aprendizes, a regularização (valor lambda) e opções de otimização incluindo o número de camadas, o tipo de otimizador clássico, o backend quântico, entre outros.
5. Treine o classificador quântico no conjunto de treinamento usando a ação `fit` da função, passando o conjunto de treinamento rotulado e o conjunto de validação, se aplicável.
6. Faça previsões no conjunto de teste nunca visto anteriormente usando a ação `predict` da função.
## Abordagem baseada em ações
A função usa uma abordagem baseada em ações. Você pode pensar nela como um ambiente virtual onde você usa ações para realizar tarefas ou alterar seu estado. Atualmente, ela oferece as seguintes ações: [list](#1-list), [create](#2-create), [delete](#3-delete), [fit](#4-fit), [predict](#5-predict), [fit_predict](#6-fit-predict) e [create_fit_predict](#7-create-fit-predict). O exemplo a seguir demonstra a ação `create_fit_predict`.

In [ ]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [1, 0, 0, 1, 0]
Probabilities (first five results):  [[0.16849563539001172, 0.8315043646099888], [0.8726393386620336, 0.12736066133796647], [0.795344837290717, 0.20465516270928288], [0.36822585748882725, 0.6317741425111725], [0.6656662698604361, 0.3343337301395641]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 7.945035696029663}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 82.41029238700867}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 77.3459484577179}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 71.27004957199097}}


### 1. List

The `list` action retrieves all stored classifiers in `*.pkl.tar` format from the shared data directory. You can also access the contents of this directory by using the `catalog.files()` method. In general, the list action searches for files with the `*.pkl.tar` extension in the shared data directory and returns them in a list format.

#### Inputs

|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | The name of the action from among `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` and `delete`. | Yes |

#### Usage

In [ ]:
job = singularity.run(action="list")

### 1. List

A ação `list` recupera todos os classificadores armazenados no formato `*.pkl.tar` do diretório de dados compartilhado. Você também pode acessar o conteúdo desse diretório usando o método `catalog.files()`. Em geral, a ação list pesquisa arquivos com a extensão `*.pkl.tar` no diretório de dados compartilhado e os retorna em formato de lista.

#### Entradas

|     Nome    |    Tipo     | Descrição |   Obrigatório  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | O nome da ação, entre `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` e `delete`. | Sim |

#### Uso

In [ ]:
job = singularity.run(
    action="create",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
)

### 2. Create
A ação `create` cria um classificador do tipo `quantum_classifier` especificado usando os parâmetros fornecidos e o salva no diretório de dados compartilhado.

> **Note:** A função atualmente suporta apenas o `QuantumEnhancedEnsembleClassifier`.
#### Entradas
|     Nome    |    Tipo     | Descrição | Obrigatório  | Padrão |
|-------------|-------------|-------------|-----------|---------|
| `action` | `str` | O nome da ação, entre `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` e `delete`. | Sim | - |
| `name` | `str` | O nome do classificador quântico, p. ex., `spam_classifier`. | Sim | - |
| `instance` | `str` | Instância IBM. | Sim | - |
| `backend_name` | `str` | Recurso de computação IBM. O padrão é `None`, o que significa que o backend com menos jobs pendentes será usado. | Não | `None` |
| `quantum_classifier` | `str` | O tipo do classificador quântico, ou seja, `QuantumEnhancedEnsembleClassifier`. | Não | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | O número de aprendizes no ensemble. | Não | `10` |
| `learners_types` | `list` | Tipos de aprendizes. Entre os tipos suportados estão: `DecisionTreeClassifier`, `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier` e `LogisticRegression`. Mais detalhes sobre cada um podem ser encontrados na [documentação do scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Não | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | Proporções de cada tipo de aprendiz no ensemble. | Não | `[1.0]` |
| `learners_options` | `list` | Opções para cada tipo de aprendiz no ensemble. Para uma lista completa de opções correspondentes ao(s) tipo(s) de aprendiz escolhido(s), consulte a [documentação do scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Não | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` ou `list` | Tipo(s) de regularização a ser(em) usado(s): `onsite` ou `alpha`. `onsite` controla o termo onsite onde valores mais altos levam a ensembles mais esparsos. `alpha` controla a compensação entre os termos de interação e onsite, onde valores mais baixos levam a ensembles mais esparsos. Se uma lista for fornecida, modelos serão treinados para cada tipo e o de melhor desempenho será selecionado. | Não | `onsite` |
| `regularization` | `str` ou `float` ou `list` | Valor de regularização. Limitado entre `0` e `+inf` se regularization_type for `onsite`. Limitado entre `0` e `1` se regularization_type for `alpha`. Se definido como `auto`, a auto-regularização é usada — o parâmetro de regularização ideal é encontrado por busca binária com a proporção desejada de classificadores selecionados para classificadores totais (`regularization_desired_ratio`) e o limite superior para o parâmetro de regularização (`regularization_upper_bound`). Se uma lista for fornecida, modelos serão treinados para cada valor e o de melhor desempenho será selecionado. | Não | `0.01` |
| `regularization_desired_ratio` | `float` ou `list` | Proporção(ões) desejada(s) de classificadores selecionados para classificadores totais para auto-regularização. Se uma lista for fornecida, modelos serão treinados para cada proporção e o de melhor desempenho será selecionado. | Não | `0.75` |
| `regularization_upper_bound` | `float` ou `list` | Limite(s) superior(es) para o parâmetro de regularização ao usar auto-regularização. Se uma lista for fornecida, modelos serão treinados para cada limite superior e o de melhor desempenho será selecionado. | Não | `200` |
| `weight_update_method` | `str` | Método para atualização dos pesos das amostras, entre `logarithmic` e `quadratic`. | Não | `logarithmic` |
| `sample_scaling` | `boolean` | Se o escalonamento de amostras deve ser aplicado. | Não | `False` |
| `prediction_scaling` | `float` | Fator de escalonamento para previsões. | Não | `None` |
| `optimizer_options` | `dictionary` | Opções do otimizador QAOA. Uma lista de opções disponíveis é apresentada mais adiante nesta documentação. | Não | ... |
| `voting` | `str` | Usar votação por maioria (`hard`) ou média de probabilidades (`soft`) para agregar previsões/probabilidades dos aprendizes. | Não | `hard` |
| `prob_threshold` | `float` | Limite de probabilidade ideal. | Não | `0.5` |
| `random_state` | `integer` | Controlar a aleatoriedade para reprodutibilidade. | Não | `None` |


- Adicionalmente, as `optimizer_options` estão listadas como segue:

|     Nome    |    Tipo     | Descrição | Obrigatório  | Padrão |
|-------------|-------------|-------------|-----------|---------|
| `num_solutions` | `integer` | O número de soluções | Não | `1024` |
| `reps` | `integer` | O número de repetições | Não | `4` |
| `sparsify` | `float` | O limiar de esparsificação | Não | `0.001` |
| `theta` | `float` | O valor inicial de theta, um parâmetro variacional do QAOA | Não | `None` |
| `simulator` | `boolean` | Se deve usar um simulador ou um QPU | Não | `False` |
| `classical_optimizer` | `str` | Nome do otimizador clássico para o QAOA. Todos os solvers oferecidos pelo SciPy, conforme listados [aqui](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize), são utilizáveis. Você precisará configurar `classical_optimizer_options` adequadamente | Não | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | Opções do otimizador clássico. Para uma lista completa de opções disponíveis, consulte a [documentação do SciPy](https://docs.scipy.org/doc/scipy/reference/) | Não | `{"maxiter": 60}` |
| `optimization_level` | `integer` | A profundidade do Circuit QAOA | Não | `3` |
| `num_transpiler_runs` | `integer` | Número de execuções do Transpiler | Não | `30` |
| `pass_manager_options` | `dictionary` | Opções para geração do gerenciador de passes pré-definido | Não | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | Opções do Estimator. Para uma lista completa de opções disponíveis, consulte a [documentação do Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) | Não | `None` |
| `sampler_options` | `dictionary` | Opções do Sampler. Para uma lista completa de opções disponíveis, consulte a [documentação do Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options) | Não | `None` |

- Os `estimator_options` padrão são:

|     Nome    |    Tipo     | Valor  |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- Os `sampler_options` padrão são:

|     Nome    |    Tipo     | Valor |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### Uso

In [ ]:
job = singularity.run(
    action="delete",
    name="classifier_name",  # specify the name of the classifier to delete here
)

#### Validações
- `name`:
    - O nome deve ser único, uma string de até 64 caracteres.
    - Pode conter apenas caracteres alfanuméricos e underscores.
    - Deve começar com uma letra e não pode terminar com underscore.
    - Nenhum classificador com o mesmo nome deve já existir no diretório de dados compartilhado.
### 3. Delete
A ação `delete` remove um classificador do diretório de dados compartilhado.

#### Entradas
|     Nome    |    Tipo     | Descrição |   Obrigatório  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | O nome da ação. Deve ser `delete`. | Sim |
| `name`      | `str`    | O nome do classificador a ser excluído. | Sim |

#### Uso

In [ ]:
job = singularity.run(
    action="fit",
    name="classifier_name",  # specify the name of the classifier to train here
    X=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
)

#### Validações
- `name`:
    - O nome deve ser único, uma string de até 64 caracteres.
    - Pode conter apenas caracteres alfanuméricos e underscores.
    - Deve começar com uma letra e não pode terminar com underscore.
    - Um classificador com o mesmo nome já deve existir no diretório de dados compartilhado.
### 4. Fit
A ação `fit` treina um classificador usando os dados de treinamento fornecidos.

#### Entradas
|     Nome    |    Tipo     | Descrição |   Obrigatório  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | O nome da ação. Deve ser `fit`. | Sim |
| `name`      | `str`    | O nome do classificador a ser treinado. | Sim |
| `X`         | `array` ou `list` ou `str` | Os dados de treinamento. Pode ser um array NumPy, uma lista ou uma string referenciando um nome de arquivo no diretório de dados compartilhado. | Sim |
| `y`         | `array` ou `list` ou `str` | Os valores alvo de treinamento. Pode ser um array NumPy, uma lista ou uma string referenciando um nome de arquivo no diretório de dados compartilhado. | Sim |
| `fit_params`| `dictionary`| Parâmetros adicionais para passar ao método `fit` do classificador. | Não |

##### fit_params
|     Nome    |    Tipo     | Descrição |   Obrigatório  |   Padrão   |
|-------------|-------------|-------------|-------------|-------------|
| `validation_data` | `tuple` | Os dados de validação e rótulos. | Não | `None` |
| `pos_label` | `integer` ou `str` | O rótulo de classe a ser mapeado para 1. | Não | `None` |
| `optimization_data` | `str` | Conjunto de dados para otimizar o ensemble. Pode ser um de: `train`, `validation`, `both`. | Não | `train` |

#### Uso

In [ ]:
job = singularity.run(
    action="predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X=X_test,  # or "X_test.npy" if you uploaded it to the shared data directory
    options={
        "out": "output.json",
    },
)

#### Validações
- `name`:
    - O nome deve ser único, uma string de até 64 caracteres.
    - Pode conter apenas caracteres alfanuméricos e underscores.
    - Deve começar com uma letra e não pode terminar com underscore.
    - Um classificador com o mesmo nome já deve existir no diretório de dados compartilhado.
### 5. Predict
A ação `predict` é usada para obter previsões hard e soft (probabilidades).

#### Entradas
|     Nome    |    Tipo     | Descrição |   Obrigatório  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | O nome da ação. Deve ser `predict`. | Sim |
| `name`      | `str`    | O nome do classificador a ser usado. | Sim |
| `X`         | `array` ou `list` ou `str` | Os dados de teste. Pode ser um array NumPy, uma lista ou uma string referenciando um nome de arquivo no diretório de dados compartilhado. | Sim |
| `options["out"]` | `str` | O nome do arquivo JSON de saída para salvar as previsões no diretório de dados compartilhado. Se não fornecido, as previsões são retornadas no resultado do job. | Não |

#### Uso

In [ ]:
job = singularity.run(
    action="fit_predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "out": "output.json",
    },
)

#### Validações
- `name`:
    - O nome deve ser único, uma string de até 64 caracteres.
    - Pode conter apenas caracteres alfanuméricos e underscores.
    - Deve começar com uma letra e não pode terminar com underscore.
    - Um classificador com o mesmo nome já deve existir no diretório de dados compartilhado.
- `options["out"]`:
    - O nome do arquivo deve ser único, uma string de até 64 caracteres.
    - Pode conter apenas caracteres alfanuméricos e underscores.
    - Deve começar com uma letra e não pode terminar com underscore.
    - Deve ter a extensão `.json`.
### 6. Fit-predict
A ação `fit_predict` treina um classificador usando os dados de treinamento e, em seguida, o usa para obter previsões hard e soft (probabilidades).

#### Entradas
|     Nome    |    Tipo     | Descrição |   Obrigatório  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | O nome da ação. Deve ser `fit_predict`. | Sim |
| `name`      | `str`    | O nome do classificador a ser usado. | Sim |
| `X_train`   | `array` ou `list` ou `str` | Os dados de treinamento. Pode ser um array NumPy, uma lista ou uma string referenciando um nome de arquivo no diretório de dados compartilhado. | Sim |
| `y_train`   | `array` ou `list` ou `str` | Os valores alvo de treinamento. Pode ser um array NumPy, uma lista ou uma string referenciando um nome de arquivo no diretório de dados compartilhado. | Sim |
| `X_test`    | `array` ou `list` ou `str` | Os dados de teste. Pode ser um array NumPy, uma lista ou uma string referenciando um nome de arquivo no diretório de dados compartilhado. | Sim |
| `fit_params`| `dictionary`| Parâmetros adicionais para passar ao método `fit` do classificador. | Não |
| `options["out"]` | `str` | O nome do arquivo JSON de saída para salvar as previsões no diretório de dados compartilhado. Se não fornecido, as previsões são retornadas no resultado do job. | Não |

#### Uso

In [ ]:
job = singularity.run(
    action="create_fit_predict",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "save": True,
        "out": "output.json",
    },
)

#### Validações
- `name`:
    - O nome deve ser único, uma string de até 64 caracteres.
    - Pode conter apenas caracteres alfanuméricos e underscores.
    - Deve começar com uma letra e não pode terminar com underscore.
    - Um classificador com o mesmo nome já deve existir no diretório de dados compartilhado.

- `options["out"]`:
    - O nome do arquivo deve ser único, uma string de até 64 caracteres.
    - Pode conter apenas caracteres alfanuméricos e underscores.
    - Deve começar com uma letra e não pode terminar com underscore.
    - Deve ter a extensão `.json`.
### 7. Create-fit-predict
A ação `create_fit_predict` cria um classificador, treina-o usando os dados de treinamento fornecidos e, em seguida, o usa para obter previsões hard e soft (probabilidades).

#### Entradas
| Nome | Tipo | Descrição | Obrigatório |
|------|------|-------------|-----------|
| `action` | `str` | O nome da ação, entre `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` e `delete`. | Sim |
| `name` | `str` | O nome do classificador a ser usado. | Sim |
| `quantum_classifier` | `str` | O tipo do classificador, ou seja, `QuantumEnhancedEnsembleClassifier`. O padrão é `QuantumEnhancedEnsembleClassifier`. | Não |
| `X_train` | `array` ou `list` ou `str` | Os dados de treinamento. Pode ser um array NumPy, uma lista ou uma string referenciando um nome de arquivo no diretório de dados compartilhado. | Sim |
| `y_train` | `array` ou `list` ou `str` | Os valores alvo de treinamento. Pode ser um array NumPy, uma lista ou uma string referenciando um nome de arquivo no diretório de dados compartilhado. | Sim |
| `X_test` | `array` ou `list` ou `str` | Os dados de teste. Pode ser um array NumPy, uma lista ou uma string referenciando um nome de arquivo no diretório de dados compartilhado. | Sim |
| `fit_params` | `dictionary` | Parâmetros adicionais para passar ao método `fit` do classificador. | Não |
| `options["save"]` | `boolean` | Se deve salvar o classificador treinado no diretório de dados compartilhado. O padrão é `True`. | Não |
| `options["out"]` | `str` | O nome do arquivo JSON de saída para salvar as previsões no diretório de dados compartilhado. Se não fornecido, as previsões são retornadas no resultado do job. | Não |

#### Uso

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

#### Validações
- `name`:
    - Se `options["save"]` estiver definido como `True`:
        - O nome deve ser único, uma string de até 64 caracteres.
        - Pode conter apenas caracteres alfanuméricos e underscores.
        - Deve começar com uma letra e não pode terminar com underscore.
        - Nenhum classificador com o mesmo nome deve já existir no diretório de dados compartilhado.

- `options["out"]`:
    - O nome do arquivo deve ser único, uma string de até 64 caracteres.
    - Pode conter apenas caracteres alfanuméricos e underscores.
    - Deve começar com uma letra e não pode terminar com underscore.
    - Deve ter a extensão `.json`.
---
## Primeiros passos
Autentique-se usando sua [chave de API da IBM Quantum Platform](http://quantum.cloud.ibm.com/) e selecione a Qiskit Function da seguinte forma:

In [3]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[-0.99958218  0.02890441]
 [ 0.03285169  0.24578719]
 [ 1.13127903 -0.49134546]
 [ 1.86951286  0.00608971]
 [ 0.20190413  0.97940529]
 [ 0.8831311   0.46912627]
 [-0.10819442  0.99412975]
 [-0.20005727  0.97978421]
 [-0.78775705  0.61598607]
 [ 1.82453236 -0.0658148 ]]
Targets: [0 1 1 1 0 0 0 0 0 1]


## Exemplo
Neste exemplo, você vai usar a função "Singularity Machine Learning - Classification" para classificar um conjunto de dados composto por dois semicírculos entrelaçados em formato de lua. O conjunto de dados é sintético, bidimensional e rotulado com rótulos binários. Ele foi criado para ser desafiador para algoritmos como clustering baseado em centróide e classificação linear.
![Conjunto de dados moons](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
Ao longo deste processo, você vai aprender como criar o classificador, ajustá-lo aos dados de treinamento, usá-lo para prever nos dados de teste e excluir o classificador quando terminar.
Antes de começar, você precisa instalar o [scikit-learn](https://scikit-learn.org/). Instale-o usando o seguinte comando:

In [4]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


Execute os seguintes passos:

1) Crie o conjunto de dados sintético usando a função [make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) do [scikit-learn](https://scikit-learn.org/).
2) Faça upload do conjunto de dados sintético gerado para o [diretório de dados compartilhado](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html).
3) Crie o classificador aprimorado quanticamente usando a ação [create](#2-create).
4) Liste seus classificadores usando a ação [list](#1-list).
5) Treine o classificador nos dados de treinamento usando a ação [fit](#4-fit).
6) Use o classificador treinado para prever nos dados de teste usando a ação [predict](#5-predict).
7) Exclua o classificador usando a ação [delete](#3-delete).
8) Limpe ao terminar.
**Passo 1.** Importe os módulos necessários e gere o conjunto de dados sintético, depois divida-o em conjuntos de treinamento e teste.

In [5]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [6]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}
['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar', 'my_classifier.pkl.tar']


**Passo 2.** Salve os conjuntos de dados de treinamento e teste rotulados no seu disco local e, em seguida, faça upload deles para o [diretório de dados compartilhado](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html).

In [7]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 8.45469617843628}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 69.4949426651001}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 73.01881957054138}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 75.4787163734436}}}}


**Step 5.** Obtain predictions and probabilities from the quantum-enhanced classifier using the [predict](#5-predict) action.

In [8]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 1, 0, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 0.0], [0.0, 1.0]]


**Passo 3.** Crie um classificador aprimorado quanticamente usando a ação [create](#2-create).

In [9]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [ ]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

## Benchmarks

These benchmarks show that the classifier can achieve extremely high accuracies on challenging problems. They also show that increasing the number of learners in the ensemble (number of qubits) can lead to increased accuracy.

"Classical accuracy" refers to the accuracy obtained using corresponding classical state of the art which, in this case, is an AdaBoost classifier based on an ensemble of size 75. "Quantum accuracy", on the other hand, refers to the accuracy obtained using the "Singularity Machine Learning - Classification".

| Problem | Dataset Size | Ensemble Size | Number of Qubits | Classical Accuracy | Quantum Accuracy | Improvement |
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
| Grid stability | 5000 examples, 12 features | 55 | 55 |  76% | 91% | 15% |
| Grid stability | 5000 examples, 12 features | 65 | 65 |  76% | 92% | 16% |
| Grid stability | 5000 examples, 12 features | 75 | 75 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 85 | 85 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 100 | 100 |  76% | 95% | 19% |

----

As quantum hardware evolves and scales, the implications for our quantum classifier become increasingly significant. While the number of qubits does impose limitations on the size of the ensemble that can be utilized, it does not restrict the volume of data that can be processed. This powerful capability enables the classifier to efficiently handle datasets containing millions of data points and thousands of features. Importantly, the constraints related to ensemble size can be addressed through the implementation of a large-scale version of the classifier. By leveraging an iterative outer-loop approach, the ensemble can be dynamically expanded, enhancing flexibility and overall performance. However, it's worth noting that this feature has not yet been implemented in the current version of the classifier.

## Changelog

### 4 June 2025
- Upgraded `QuantumEnhancedEnsembleClassifier` with the following updates:
  - Added onsite/alpha regularization. You can specify `regularization_type` to be `onsite` or `alpha`
  - Added auto-regularization. You can set `regularization` to `auto` to use auto-regularization
  - Added `optimization_data` parameter to the `fit` method to choose optimization data for quantum optimization. You can use one of these options: `train`, `validation`, or `both`
  - Improved overall performance
- Added detailed status tracking for running jobs

### 20 May 2025
- Standardized error handling

### 18 March 2025
- Upgraded qiskit-serverless to 0.20.0 and base image to 0.20.1

### 14 February 2025
- Upgraded base image to 0.19.1

### 6 February 2025
- Upgraded qiskit-serverless to 0.19.0 and base image to 0.19.0

### 13 November 2024
- Release of Singularity Machine Learning - Classification

## Get support

For any questions, [reach out to Multiverse Computing](mailto:singularity@multiversecomputing.com).

Be sure to include the following information:

- The Qiskit Function Job ID (`job.job_id`)
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Multiverse Computing's Singularity Machine Learning Classification function](https://quantum.cloud.ibm.com/functions?id=multiverse-singularity).
- Review [Leclerc, L., et al. (2023). Financial risk management on a neutral atom quantum processor. Physical Review Research, 5, 043117.](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.5.043117)

</Admonition>